# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one page (`content_id`), scoped to one client (`client_id`), as of the extract date.

Time windows are layered, not one single window — this is worth stating explicitly because it's a common trap:

* `*_90d` fields → trailing 90 days from extract date
* `*_last_30d` → most recent 30 days
* `*_prev_30d` → the 30 days before that (used to compute trend_direction/trend_pct)
* `content_age_days` → since page creation, not a fixed window
* `days_since_last_update` → since last edit, also not a fixed window

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Grain check: is content_id unique, or unique per (content_id, client_id)?
dupe_content_id = df["content_id"].duplicated().sum()
dupe_pair = df.duplicated(subset=["content_id", "client_id"]).sum()

print("Duplicate content_id alone:", dupe_content_id)
print("Duplicate (content_id, client_id) pairs:", dupe_pair)
print("Total rows:", len(df), "| Unique content_id:", df["content_id"].nunique())

Duplicate content_id alone: 0
Duplicate (content_id, client_id) pairs: 0
Total rows: 30000 | Unique content_id: 30000


Since `dupe_content_id` is 0, the grain is simply "one row per page" — no need for the client pair.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Context** (grouping/joining only, never fed to the model):

* `content_id` — page identifier
* `client_id` — client identifier

**Feature** (structured, knowable at analysis time, safe to cluster on):

* Demand/economics: `search_volume`,`competition`, `competition_level`, `cpc`
* Content shape: `word_count`, `char_count`, `word_count_tier`, `char_count_tier`, `content_type`, `main_intent`
* Performance (90d): `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`
* Derived rates: `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`
* Age/freshness: `content_age_days`, `age_tier`, `age_tier_order`, `days_since_last_update`, `freshness_tier`
* Tiers: `impression_tier`, `position_tier`

**Label** / **proxy**: **none**. This is unsupervised clustering — say that plainly rather than forcing something into this bucket. The one nuance worth a line in your write-up: `trend_direction` and `trend_pct` are derived from `*_last_30d` vs `*_prev_30d` (a defined rule, not an observed future outcome). That's fine to use as a feature for clustering, but flag it — if you later cluster, then "discover" that one cluster is declining, and present that as insight, that's circular: you fed the conclusion in as an input. Use trend as a feature for grouping, but don't claim it as a novel finding of the clustering itself.

Excluded:

* `provider_used`, `model_used` — production/tooling metadata (which AI provider or model generated the content). Excluded because it describes how the page was made, not how it performs; mixing it into performance clustering risks the clusters reflecting production pipeline rather than audience behavior. (Judgment call — you could argue this back in as a feature if your research question is specifically about provider-performance differences, but that's a different question than archetype clustering.)

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
# --- Counts ---
print("Total rows:", len(df))
print("Unique clients:", df["client_id"].nunique())
print("Rows per client (describe):")
print(df.groupby("client_id").size().describe())

# --- Missingness: overall ---
missing = df.isna().mean().sort_values(ascending=False)
print("\nColumns with any missing values:\n", missing[missing > 0])

# --- Missingness: does it follow a pattern (e.g. by content_type)? ---
if missing[missing > 0].shape[0] > 0:
    top_missing_col = missing[missing > 0].index[0]
    print(f"\nMissingness of '{top_missing_col}' by content_type:")
    print(df.groupby("content_type")[top_missing_col].apply(lambda s: s.isna().mean()))

# --- Window sanity: do last_30d + prev_30d roughly reconcile with 90d? ---
window_check = df[["impressions_last_30d", "impressions_prev_30d", "impressions_90d"]].copy()
window_check["sum_60d"] = window_check["impressions_last_30d"] + window_check["impressions_prev_30d"]
window_check["remainder_30d"] = window_check["impressions_90d"] - window_check["sum_60d"]
print("\nDoes 90d - (last_30d + prev_30d) look like a sane 'oldest 30d' remainder?")
print(window_check["remainder_30d"].describe())

# --- content_age_days vs days_since_last_update sanity ---
bad_freshness = (df["days_since_last_update"] > df["content_age_days"]).sum()
print("\nRows where days_since_last_update > content_age_days (should be 0):", bad_freshness)

Total rows: 30000
Unique clients: 32
Rows per client (describe):
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
dtype: float64

Columns with any missing values:
 provider_used        0.714600
word_count           0.256633
char_count           0.256633
word_count_tier      0.256633
char_count_tier      0.256633
model_used           0.191100
trend_pct            0.112933
competition_level    0.087000
search_volume        0.082267
cpc                  0.082267
competition          0.082267
main_intent          0.079133
scroll_rate          0.004167
dtype: float64

Missingness of 'provider_used' by content_type:
content_type
comparison article    0.833572
feedly article        0.700382
keyword article       0.712647
Name: provider_used, dtype: float64

Does 90d - (last_30d + prev_30d) look like a sane 'oldest 30d' remainder?
count     30000.000000
mean       1988.229067


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Say plainly what this data can never answer, so nobody downstream over-reads the clusters:

* **No article text**. Clustering here is behavioral/structural, not semantic — it groups by traffic and engagement shape, not by topic or meaning. A cluster labeled "hidden gem" says nothing about why the content resonates.
* **Trailing windows only, no long history**. `*_90d` and `*_last_30d`/`*_prev_30d` give a snapshot, not a trend over the page's full life. A page could look "stable" simply because the 90-day window missed an earlier spike or crash.
* `trend_direction`/`trend_pct` **is a defined rule, not an observed future outcome**. It compares two adjacent 30-day windows — useful as a feature, but it's not a validated forecast of what happens next, and shouldn't be reframed as "predicted" performance anywhere in your write-up.
* **Uneven page age**. `content_age_days` varies a lot across the inventory (worth an actual `.describe()` in your notebook) — very young pages haven't had time to accumulate the same signal older pages have, so "low traffic" for a 20-day-old page means something different than for a 2-year-old page. Clustering on raw volume without accounting for age can just rediscover "old vs new," not real archetypes.
* **No causal or client-context signal**. Nothing here explains why a page underperforms (no info on backlinks, internal linking, competitor moves, seasonality, or business priority) — clusters describe correlated behavior, not causes, and definitely not a client's business reasons for a page existing.
* **Provider/model metadata excluded from clustering**, so this analysis can't speak to whether one AI provider or model produces better- or worse-performing content — that's a legitimate separate question, just not this one.

That closes the contract — one row = one page (pending your dupe check), features listed, no label since it's unsupervised, every claim backed by a query cell, and the limits stated up front so nobody mistakes a behavioral cluster for a semantic or causal one.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.